# 第64课 · 训练评估闭环 — train loop + val loop，准确率 / 混淆矩阵（confusion matrix，CM） / 过拟合（overfitting）诊断

**目标**：实现 `train_epoch` 和 `eval_accuracy`，完整训练 KeywordCNN 5 个 epoch，画出损失曲线与混淆矩阵。

🔗 **Aurora 连接**：
- mel 特征由 `aurora.audio.mfcc`（STFT → mel filterbank → log）提供，详见 `src/aurora/audio/`
- 这是 Aurora 项目第一个实际训练完毕的模型，后续推理模块直接加载其权重


← **上一课**　[L63 · 音频分类模型](L63_kws_model.ipynb)

> 上节课学习了 **音频分类模型**：CNN + Mel 特征，在 Speech Commands 上定义网络。  
> 本课将探讨 **训练评估闭环**。

## 本课剧情：为什么"看 loss 下降"不够，还要看混淆矩阵？

你训练了一个关键词识别器，loss 从 2.3 降到 0.5，准确率 90%——听起来不错？

**问题**：若数据集里 "yes" 占 80%，模型只要每次都猜 "yes"，准确率就有 80%。但这根本不是真正的识别，是**样本不均衡**导致的假象。

**三个诊断工具**：

| 工具 | 看什么 | 发现什么问题 |
|---|---|---|
| 训练 loss | 模型是否在学习 | 过拟合/欠拟合 |
| 验证准确率 | 泛化能力 | 整体判断，但掩盖不均衡 |
| 混淆矩阵 | 每对 (真实,预测) 的样本数 | 哪些类互相混淆 |

**交叉熵损失**（`nn.CrossEntropyLoss`）：接受 logit（未经 softmax 的原始输出），内部自动算 softmax 再算 log loss：
```
CE = -log(softmax(logit)[y])    ← 正确类的对数概率取负
初始值 ≈ log(num_classes) = log(10) ≈ 2.3   ← 随机猜测的期望损失
```

**训练闭环 = 两个函数**：
```
train_epoch(model, loader, optimizer, criterion) → avg_loss
eval_accuracy(model, loader)                      → accuracy %
```

本节任务：实现这两个函数，让训练循环能在 5 轮内看到 loss 下降、准确率上升。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 1. 交叉熵损失 与 Adam 优化器

**交叉熵损失**（`nn.CrossEntropyLoss`）输入原始 logit（形状 `[B, C]`）和整数标签（形状 `[B]`），内部做 softmax 再取负对数似然：

```
loss = -log( exp(logit[y]) / sum_j exp(logit[j]) )
```

不要在模型末尾手动加 softmax——`CrossEntropyLoss` 已经包含了。

**Adam 优化器** 结合了动量（一阶矩）和自适应学习率（二阶矩）。关键词识别（keyword spotting，KWS）任务通常以学习率 `lr=1e-3` 作为起点。

```python
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
```

In [ ]:
# 演示：CrossEntropyLoss 的行为
criterion_demo = nn.CrossEntropyLoss()

# 3 个样本，4 类
logits = torch.tensor([[2.0, 0.5, 0.1, 0.1],   # 自信预测类 0
                        [0.1, 0.1, 2.0, 0.1],   # 自信预测类 2
                        [0.5, 0.5, 0.5, 0.5]])  # 完全不确定
labels = torch.tensor([0, 2, 1])  # 真实标签

loss_val = criterion_demo(logits, labels)
print(f'损失值: {loss_val.item():.4f}')
print('前两个样本预测正确且自信 → 损失小；第三个不确定且错误 → 损失大')

## 2. Per-class Accuracy（每类准确率）

整体准确率 `correct / total` 掩盖了类别不均衡的问题。Per-class accuracy 分别统计每个关键词类别：

```
accuracy_c = (预测为 c 且真实为 c 的样本数) / (真实为 c 的样本总数)
```

如果某类占数据集 1%，整体 99% 准确率可能完全靠忽略它得到；per-class 准确率会暴露这个问题。

In [ ]:
# 演示：手动计算 per-class accuracy
y_true = torch.tensor([0, 0, 1, 1, 2, 2, 2])
y_pred = torch.tensor([0, 1, 1, 1, 0, 2, 2])  # 类0预测错一个，类2预测错一个

num_classes = 3
for c in range(num_classes):
    mask = (y_true == c)
    acc_c = (y_pred[mask] == c).float().mean().item()
    print(f'  类 {c}: {acc_c:.2%}')

overall = (y_pred == y_true).float().mean().item()
print(f'整体准确率: {overall:.2%}')

## 3. 混淆矩阵

混淆矩阵 `C` 的形状为 `[num_classes, num_classes]`：

```
C[i][j] = 真实标签为 i、预测为 j 的样本数
```

- **对角线** `C[i][i]`：正确预测
- **非对角线** `C[i][j]`（i≠j）：错误预测（把类 i 误判为类 j）

用热力图可视化时，颜色越深代表该格样本越多；理想情况下热力图只有对角线是深色的。

In [ ]:
# 演示：用 sklearn 计算并可视化混淆矩阵
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true_np = y_true.numpy()
y_pred_np = y_pred.numpy()
class_names = ['yes', 'no', 'stop']

cm = confusion_matrix(y_true_np, y_pred_np)
print('混淆矩阵:')
print(cm)

fig, ax = plt.subplots(figsize=(4, 3))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('混淆矩阵示例')
plt.tight_layout()
plt.show()

## 4. ✏️ 实现 `train_epoch(model, loader, optimizer, criterion)`

**5步实现（每个 batch 重复）**：

| 步骤 | 操作 | 注意 |
|---|---|---|
| 1 | `model.train()` | 开启 dropout/BN 训练模式（放在 epoch 开始） |
| 2 | `optimizer.zero_grad()` | 清零梯度，每 batch 必须执行 |
| 3 | `loss = criterion(model(x), y)` | 前向 + 计算 loss |
| 4 | `loss.backward()` | 反向传播 |
| 5 | `optimizer.step()` | 更新参数 |

**返回值**：`avg_loss: float` = 所有 batch 的平均损失

**验收标准**：
- 函数返回 float，不是 Tensor
- 多次调用 loss 应下降（对合理的 lr 和数据）

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    """训练一个 epoch，返回平均损失。"""
    # ✏️ TODO: model.train()
    # ✏️ TODO: for x, y in loader:
    #              optimizer.zero_grad()
    #              out = model(x)
    #              loss = criterion(out, y)
    #              loss.backward()
    #              optimizer.step()
    #              累积 loss.item()
    # ✏️ TODO: return avg_loss
    raise NotImplementedError

In [ ]:
# 目视验证 train_epoch
# 用一个极小的假模型 + 假 loader 快速冒烟测试
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
_x = torch.randn(16, 1, 40, 32)   # 16 个样本，单通道 mel 图
_y = torch.randint(0, 4, (16,))   # 4 类随机标签
_loader = DataLoader(TensorDataset(_x, _y), batch_size=8)

_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(40*32, 4)
)
_opt = torch.optim.Adam(_model.parameters(), lr=1e-3)
_crit = nn.CrossEntropyLoss()

_loss = train_epoch(_model, _loader, _opt, _crit)
assert isinstance(_loss, float), '返回值应为 float'
assert _loss > 0, '损失应大于 0'
print(f'✅ train_epoch 通过，avg_loss = {_loss:.4f}')

# 增强验证：确认权重确实被更新（第 2 epoch 损失应低于第 1 epoch）
_loss_e2 = train_epoch(_model, _loader, _opt, _crit)
assert _loss_e2 < _loss, (
    f'第 2 epoch 损失({_loss_e2:.4f})应小于第 1 epoch({_loss:.4f})；'
    '请检查 optimizer.step() 是否被正确调用，以及 zero_grad() 是否在每批前执行'
)
print(f'✅ 权重更新验证通过：epoch1={_loss:.4f} → epoch2={_loss_e2:.4f}')

## 5. ✏️ 实现 `eval_accuracy(model, loader)`

**3步实现**：

| 步骤 | 操作 | 关键细节 |
|---|---|---|
| 1 | `model.eval()` + `torch.no_grad():` | 关闭 dropout，禁止梯度计算（省显存）|
| 2 | `preds = model(x).argmax(dim=1)` | logit 最大值索引 = 预测类别 |
| 3 | `correct += (preds == y).sum()` | 累积正确数，最后 / total |

**返回值**：`accuracy: float` ∈ [0.0, 1.0]

**验收标准**：
- 返回 float（不是 Tensor）
- 随机初始化时 ≈ 1/num_classes（随机猜测）
- 训练后应 > 随机基线

In [ ]:
def eval_accuracy(model, loader):
    """在验证集上计算整体准确率。"""
    # ✏️ TODO: model.eval()
    # ✏️ TODO: with torch.no_grad():
    #              for x, y in loader:
    #                  preds = model(x).argmax(dim=1)
    #                  累积 correct / total
    # ✏️ TODO: return correct / total
    raise NotImplementedError

In [ ]:
# 目视验证 eval_accuracy
_acc_before = eval_accuracy(_model, _loader)
assert 0.0 <= _acc_before <= 1.0, '准确率应在 [0, 1]'
print(f'✅ eval_accuracy 通过，准确率 = {_acc_before:.2%}')

# 增强验证：边界检查——完美预测模型应返回准确率 1.0
_bnd_x = torch.zeros(4, 1, 40, 32)
_bnd_y = torch.tensor([0, 1, 2, 3])
_bnd_loader = DataLoader(TensorDataset(_bnd_x, _bnd_y), batch_size=4)

class _PerfectClassifier(nn.Module):
    """总是正确预测的占位模型：第 i 个样本预测类 i。"""
    def forward(self, x):
        # 4 样本 4 类，torch.eye(4) 使每个样本的正确类 logit 最大
        return torch.eye(4)

_acc_perfect = eval_accuracy(_PerfectClassifier(), _bnd_loader)
assert abs(_acc_perfect - 1.0) < 1e-6, (
    f'完美预测应返回 1.0，实际得到 {_acc_perfect:.4f}；'
    '请检查 (preds == y).sum() 的统计逻辑'
)
print(f'✅ 边界检查通过：完美预测准确率 = {_acc_perfect:.2%}')

## 6. 参数实验：训练曲线 + 混淆矩阵

**实验参数**：
- `num_epochs = 5`：观察损失从初始值（≈log(num_classes)≈1.39）下降的曲线形状
- `lr = 1e-3`（Adam 默认）：可改为 `1e-4` 对比收敛速度
- `batch_size = 8`：小批量训练，损失曲线会有抖动

**预期现象**：
- 损失曲线前 2 epoch 下降最快，之后趋于平缓
- 混淆矩阵对角线逐渐变深，非对角线变淡
- 若某两类混淆严重（非对角格颜色深），说明它们的 mel 特征相似，可能需要更多训练数据或更深的网络

In [ ]:
# 参数实验：5 epoch 训练曲线 + 混淆矩阵
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 重新初始化模型（用假数据演示，替换为真实 KeywordCNN + DataLoader 即可）
torch.manual_seed(0)
exp_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(40*32, 64),
    nn.ReLU(),
    nn.Linear(64, 4)
)
exp_opt = torch.optim.Adam(exp_model.parameters(), lr=1e-3)
exp_crit = nn.CrossEntropyLoss()

# 假数据 loader（真实使用时替换为 SpeechCommandsDataset 的 DataLoader）
torch.manual_seed(1)
_x_big = torch.randn(128, 1, 40, 32)
_y_big = torch.randint(0, 4, (128,))
# ✅ 正确做法：用 random_split 把数据集切成独立的训练集和验证集
# 若 val_loader 和 train_loader 使用同一份数据，"验证准确率"等于训练准确率，
# 模型永远无法展示泛化间隔，过拟合诊断也就无从谈起。
from torch.utils.data import random_split
_full_ds = TensorDataset(_x_big, _y_big)
_train_ds, _val_ds = random_split(_full_ds, [100, 28],
                                  generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(_train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(_val_ds,   batch_size=16)

num_epochs = 10
train_losses = []
val_accs = []

for epoch in range(1, num_epochs + 1):
    loss = train_epoch(exp_model, train_loader, exp_opt, exp_crit)
    acc  = eval_accuracy(exp_model, val_loader)
    train_losses.append(loss)
    val_accs.append(acc)
    print(f'Epoch {epoch}/{num_epochs}  loss={loss:.4f}  acc={acc:.2%}')

# 画训练曲线
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(range(1, num_epochs+1), train_losses, marker='o', color='steelblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('训练损失曲线')
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, num_epochs+1), val_accs, marker='s', color='tomato')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('验证准确率曲线')
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 混淆矩阵
exp_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        all_preds.append(exp_model(xb).argmax(dim=1))
        all_labels.append(yb)
all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

cm = confusion_matrix(all_labels, all_preds)
class_names = ['yes', 'no', 'stop', 'go']

fig2, ax2 = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax2, colorbar=True, cmap='Blues')
ax2.set_title('验证集混淆矩阵（5 epoch 后）')
plt.tight_layout()
plt.show()

## 本课收束

本节实现了 `train_epoch`（返回 `avg_loss: float`）和 `eval_accuracy`（返回 `accuracy: float`），构成了 KeywordCNN 完整的监督训练闭环。mel 特征由 `aurora.audio`（`src/aurora/audio/`）中的手写 STFT → mel filterbank 流水线提供，这是 Aurora 项目第一个端到端训练完毕的神经网络模型。下一节（L65）将对训练过程做深度可视化：实时绘制 Loss/Acc 曲线、监控梯度范数与权重分布直方图，帮助诊断过拟合、梯度消失等常见训练问题。

> **关于过拟合诊断范围**：本课（L64）通过独立验证集（cell-16 的 `random_split`）建立了观察过拟合的基础结构——训练损失持续下降而验证准确率趋于平稳。L65 将提供系统量化工具（early stopping 判据、梯度范数监控等），两课合看完整覆盖"过拟合介绍 → 深度诊断"链路。

## ✏️ 白板挑战：训练评估闭环手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：`nn.CrossEntropyLoss` 随机初始化模型时期望损失是多少？（10类分类）  
（log(10)≈2.303，softmax输出≈均匀分布，每类概率≈0.1，-log(0.1)=2.303）

**问 2**：为什么 `eval_accuracy` 必须用 `torch.no_grad()` 而 `train_epoch` 不用？  
（推理不需要梯度，no_grad减少显存占用和计算量；训练需要梯度用于backward）

**问 3**：`model.train()` 和 `model.eval()` 对哪些层有影响？  
（Dropout: train时随机丢弃，eval时全部保留；BatchNorm: train用batch统计，eval用running mean/var）

**问 4**：混淆矩阵 `C[i][j]` 的含义是什么？对角线代表什么？  
（C[i][j]=真实类别i被预测为j的样本数；对角线C[i][i]=正确预测数；理想时只有对角线非零）

**问 5**：若 `eval_accuracy` 返回了一个 Tensor 而不是 float，后续 `print(f"acc={acc:.4f}")` 会怎样？  
（Tensor支持格式化打印，但在不同上下文可能触发梯度计算或设备错误；应用.item()转float）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import torch, torch.nn as nn

# 问1：随机模型CE loss ≈ log(num_classes)
criterion_q = nn.CrossEntropyLoss()
n_classes = 10
logits_q = torch.zeros(100, n_classes)   # 均匀logit
labels_q = torch.randint(0, n_classes, (100,))
loss_q = criterion_q(logits_q, labels_q)
expected = torch.log(torch.tensor(float(n_classes)))
assert abs(loss_q.item() - expected.item()) < 0.01, f"loss={loss_q.item():.4f}"
print(f"Q1 ✅  均匀logit下CE={loss_q.item():.4f}≈log({n_classes})={expected.item():.4f}")

# 问2：no_grad减少显存
x_q = torch.randn(4, 1, 40, 81, requires_grad=True)
with torch.no_grad():
    y_q = x_q * 2
assert y_q.grad_fn is None
print(f"Q2 ✅  no_grad内的运算grad_fn={y_q.grad_fn}（无计算图，节省显存）")

# 问3：train/eval模式对Dropout的影响
import torch.nn.functional as F
torch.manual_seed(0)
dp = nn.Dropout(p=0.5)
x_test = torch.ones(100)
dp.train(); out_train = dp(x_test)
dp.eval();  out_eval  = dp(x_test)
train_zeros = (out_train == 0).sum().item()
eval_zeros  = (out_eval  == 0).sum().item()
assert eval_zeros == 0, "eval模式Dropout不丢弃"
assert train_zeros > 0, "train模式Dropout应丢弃部分"
print(f"Q3 ✅  Dropout: train模式丢弃{train_zeros}/100个，eval模式丢弃{eval_zeros}/100个")

# 问4：混淆矩阵对角线
cm_q = torch.zeros(3, 3, dtype=torch.long)
preds_q = torch.tensor([0, 0, 1, 1, 2, 2])
labels_q2 = torch.tensor([0, 1, 1, 2, 2, 0])
for p, l in zip(preds_q, labels_q2):
    cm_q[l, p] += 1
diagonal_sum = cm_q.diagonal().sum().item()
total_q2 = len(preds_q)
print(f"Q4 ✅  混淆矩阵对角线之和={diagonal_sum}={diagonal_sum}/{total_q2}正确")
print(f"      C[i][j]=真实i预测为j: {cm_q.tolist()}")

# 问5：Tensor vs float返回值
t_acc = torch.tensor(0.85)
f_acc = t_acc.item()
assert isinstance(f_acc, float)
print(f"Q5 ✅  .item()将Tensor({t_acc.dtype})转为Python float={f_acc:.4f}")
print("\n🎉 训练评估闭环白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l64_review = {
    "cross_entropy_understood": None,  # 理解CE初始值≈log(num_classes)，随机猜测的期望损失？True/False
    "train_epoch_impl":         None,  # train_epoch()实现正确，5步循环，返回avg_loss float？True/False
    "eval_accuracy_impl":       None,  # eval_accuracy()实现正确，model.eval()+no_grad？True/False
    "confusion_matrix_read":    None,  # 能读懂混淆矩阵，知道对角线=正确预测？True/False
    "whiteboard_passed":        None,  # 白板挑战5道推导全部完成？True/False
}

unfilled = [k for k, v in l64_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l64_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L64 全部通关！进入 L65：训练可视化')

---

→ **下一课**　[L65 · 训练可视化](L65_visual_training.ipynb)

> 下节课将学习 **训练可视化**：Loss / Acc 曲线，梯度范数，权重分布直方图。